In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, silhouette_score
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt

KMEANS WITH DIFFERENT HYPERPARAMETERS

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
root_dir = r'smaller_dataset.csv'
df = pd.read_csv(root_dir)

In [ ]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Uploading_Attack"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]  # replace with your actual column names

for col in categorical_columns:
    df[col] = df[col].astype('category')

In [ ]:
for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [ ]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':  
        return 0
    else:
        return 1

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

print(df[['Attack_Type', 'Anomaly_Label']].head())

In [ ]:
null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]  

inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
df = df.dropna()
df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
labels = df['Anomaly_Label']
attack_types = df['Attack_Type']  #to track which attacks are detected

feature_combos = [
   ['SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Flow IAT Min'],
   ['SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'SYN Flag Count', 'Flow IAT Min'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Dst Port_freq', 'Flow IAT Min', 'Flow IAT Mean'],
   ['Fwd Segment Size Avg', 'SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
   ['SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Flow IAT Min'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Dst Port_freq', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Packet Length Max', 'SYN Flag Count', 'Idle Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Dst Port_freq', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Dst Port_freq', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Packet Length Max', 'SYN Flag Count', 'Dst Port_freq', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Flow IAT Max', 'Dst Port_freq', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Max', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Packet Length Max', 'Dst Port_freq', 'Idle Mean', 'Packet Length Mean', 'Flow IAT Min'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean']
]

In [ ]:
k_values = [
    [3,4],
    [5],
    [4],
    [5,6],
    [6],
    [4,5],
    [4,5,6],
    [5],
    [4],
    [5],
    [5,6],
    [5],
    [6,7],
    [6],
    [6],
    [6,7,8],
    [6,7,8],
    [7,8],
    [7,8],
    [7,8],
    [8],
    [7],
    [6,7,8],
    [6,7,8]
]

In [ ]:
param_grid = {
    "n_init":      [10, 20],
    "init":        ["k-means++", "random"],
    "algorithm":   ["lloyd", "elkan"],
    "max_iter":    [300, 500],
}
hp_list = list(ParameterGrid(param_grid))

output_file = "kmeans_hp_results.jsonl"
with open(output_file, "w") as f_out:
    
    for i, features in enumerate(feature_combos):
        X = df[features].fillna(0)
        X_scaled = StandardScaler().fit_transform(X)

       
        for k in k_values[i]:

           
            for hp in hp_list:
                
                km = KMeans(n_clusters=k, random_state=42, **hp)
                pred_clusters = km.fit_predict(X_scaled)


                
                cluster_map = {}
                for cid in range(k):
                    idxs = np.where(pred_clusters == cid)[0]
                    if len(idxs):
                        cluster_map[cid] = np.argmax(
                            np.bincount(labels.to_numpy()[idxs])
                        )
                    else:
                        cluster_map[cid] = 0
                pred_labels = np.array([cluster_map[c] for c in pred_clusters])

               
                cm = confusion_matrix(labels, pred_labels, labels=[0,1])
                TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0,0,0,0)
                precision = TP/(TP+FP) if TP+FP else 0.0
                recall    = TP/(TP+FN) if TP+FN else 0.0
                f1_score  = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0

                
                attack_counts = {} 
                attack_hits = {} 
                
                for t,p,atk in zip(labels, pred_labels, attack_types):
                    if t==1:
                        attack_counts[atk] = attack_counts.get(atk,0) + 1
                        if p==1:
                            attack_hits[atk] = attack_hits.get(atk,0) + 1
                attack_pct = {
                    atk: round(100*attack_hits.get(atk,0)/total,2)
                    for atk,total in attack_counts.items()
                }

                
                out = {
                    "feature_combo":    features,
                    "k":                k,
                    **hp,                    
                    "inertia":          round(km.inertia_,2),
                    "precision":        round(precision,4),
                    "recall":           round(recall,4),
                    "f1_score":         round(f1_score,4),
                    "attack_detection_%": attack_pct
                }
                f_out.write(json.dumps(out) + "\n")

print(f"✅ Grid search results saved to {output_file}")


In [ ]:
import json

results = []
with open(output_file, "r") as f:
    for line in f:
        results.append(json.loads(line))

# Sort by F1 score (descending)
sorted_by_f1 = sorted(results, key=lambda x: x['f1_score'], reverse=True)

# Show top 5 results
for r in sorted_by_f1[:5]:
    print(f"\nFeatures: {r['feature_combo']}")
    print(f"k: {r['k']}")
    # print each hyperparameter
    print("Hyperparameters:")
    print(f"  init     : {r['init']}")
    print(f"  n_init   : {r['n_init']}")
    print(f"  algorithm: {r['algorithm']}")
    print(f"  max_iter : {r['max_iter']}")
    print(f"F1 Score: {r['f1_score']}")
    print(f"Precision: {r['precision']}")
    print(f"Recall: {r['recall']}")
    print(f"Total Anomaly Detection Rate: {r['recall'] * 100:.2f}%")

    # Sort all attack types by detection % and print each
    all_attacks = sorted(r['attack_detection_%'].items(), key=lambda x: x[1], reverse=True)
    print("Attack detection rates:")
    for attack, pct in all_attacks:
        print(f"  {attack}: {pct}%")
